In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pydantic import ValidationError
from typing import Literal
import json

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
class SupportTicket(BaseModel):
    category: Literal["BILLING", "TECHNICAL", "OTHER"]
    priority: Literal["LOW", "MEDIUM", "HIGH"]
    amount: float

In [ ]:
def extract_ticket(ticket_text: str) -> str | None:
    try:
        llm_response = client.responses.create(
        model="gpt-6-luna",
        input=ticket_text,
        instructions="""
            TASK:
            You are a ticketing agent which categorises ticket, assigns a priority and defines the amount if available.

            CATEGORIES:
            - category: ["BILLING", "TECHNICAL", "OTHER"]
            - priority: ["LOW", "MEDIUM", "HIGH"]
            - amount: AMOUNT | 0.0

            SCHEMA:
            - category
                - required
            - priority
                - required
            - amount
                - required
                - If no amount is specified, return 0.0
            
            OUTPUT FORMAT:
            Output should always be a JSON object.
        """
        )

        return llm_response.output_text
    
    except Exception as e:
        print(f"Model failed to return: {e}")

In [10]:
def validate_ticket(llm_response: str):
    try:
        parsed = json.loads(llm_response)
        ticket = SupportTicket.model_validate(parsed)
        return ticket

    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return

    except ValidationError as e:
        print(f"Schema validation failed: {e}")
        return

In [11]:
tickets = [
    "My payment of Rs. 2500 failed. Please fix this urgently.",

    "The application crashes whenever I try to log in.",

    "I would like to know your working hours."
]

In [ ]:
for ticket in tickets:
    llm_response = extract_ticket(ticket)
    if llm_response:
        validated = validate_ticket(llm_response)

        print(f'Ticket text: {ticket}')
        print(f'LLM respinse: {llm_response}')
        print(f'Validated: {validated}')
    else:
        print("LLM did not return answer.")

Ticket text: My payment of Rs. 2500 failed. Please fix this urgently.
LLM respinse: {"category":"BILLING","priority":"HIGH","amount":2500.0}
Validated: category='BILLING' priority='HIGH' amount=2500.0
Ticket text: The application crashes whenever I try to log in.
LLM respinse: {"category":"TECHNICAL","priority":"HIGH","amount":0.0}
Validated: category='TECHNICAL' priority='HIGH' amount=0.0
Ticket text: I would like to know your working hours.
LLM respinse: {"category":"OTHER","priority":"LOW","amount":0.0}
Validated: category='OTHER' priority='LOW' amount=0.0
